In [1]:
# =============================================================================
# Autor:   Thiago Souza (@engthiago1979-blip)  |  v4 (06-06-2026)
# Skill:   analista-python-ambiental
# Uso:     Estatística (P95, MK, KW, Dunn, ruptures, pandera, IQR) + plots.
# v4:      Os VMP deixam de ser hardcoded e passam a ser LIDOS do YAML normativo
#          (normas/conama_430_2011.yaml), fonte unica versionada e rastreavel.
#          A conformidade FORMAL (amostral) e feita na celula 4.
# =============================================================================
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.stats as stats
import pymannkendall as mk
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import yaml

warnings.filterwarnings("ignore", message="invalid value encountered")
warnings.filterwarnings("ignore", message="Precision loss occurred")
warnings.filterwarnings("ignore", module="pandera")
_ciw = getattr(stats, 'ConstantInputWarning', None)
if _ciw is not None:
    warnings.filterwarnings("ignore", category=_ciw)

try:
    import scikit_posthocs as sp
    _HAS_DUNN = True
except Exception:
    _HAS_DUNN = False
try:
    import ruptures as rpt
    _HAS_RUPT = True
except Exception:
    _HAS_RUPT = False
try:
    import pandera.pandas as pa
    _HAS_PANDERA = True
except Exception:
    try:
        import pandera as pa
        _HAS_PANDERA = True
    except Exception:
        _HAS_PANDERA = False

RAIZ = Path(r"C:\Users\thiago\Documents\estudos_python\etes_nesa")
DIR_BASE = RAIZ / "base_dados"
DIR_PROJ = RAIZ / "Efluentes_Sanitários"
DIR_PRODUTOS = DIR_PROJ / "produtos"
FILE_INPUT = DIR_BASE / "ETEs_NorteEnergia_Processado_v1.xlsx"
FILE_NORMA = DIR_PROJ / "normas" / "conama_430_2011.yaml"
DIR_PRODUTOS.mkdir(parents=True, exist_ok=True)

NESA_COLORS = {'primary':'#009CA7','secondary':'#004851','vmp':'#F25A3C','ok':'#52BD8B',
               'text':'#404041','muted':'#9aa0a6','highlight':'#B9F8FF'}
MESES_PT = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

def formatar_meses_pt(ax):
    def _fmt(x, pos=None):
        d = mdates.num2date(x)
        return f"{MESES_PT[d.month - 1]}/{d:%y}"
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(_fmt))

def configurar_estilo():
    plt.rcParams.update({'font.family':'sans-serif',
        'font.sans-serif':['Segoe UI','Calibri','Arial','DejaVu Sans'],
        'font.size':11,'axes.labelsize':11,'axes.labelcolor':NESA_COLORS['text'],
        'axes.edgecolor':'#d6d6d6','axes.linewidth':1.0,'axes.axisbelow':True,
        'axes.grid':True,'grid.color':'#ececec','grid.linewidth':0.9,
        'xtick.color':NESA_COLORS['text'],'ytick.color':NESA_COLORS['text'],
        'text.color':NESA_COLORS['text'],'figure.dpi':110,'savefig.dpi':300,
        'savefig.bbox':'tight','legend.frameon':False})
configurar_estilo()

def titulo_storytelling(ax, titulo, subtitulo=None, fonte=None):
    ax.set_title('')
    ax.text(0.0,1.13,titulo,transform=ax.transAxes,fontsize=15,fontweight='bold',color='#2b2b2b',va='bottom',ha='left')
    if subtitulo:
        ax.text(0.0,1.04,subtitulo,transform=ax.transAxes,fontsize=10.5,color=NESA_COLORS['muted'],va='bottom',ha='left')
    if fonte:
        ax.text(0.0,-0.30,fonte,transform=ax.transAxes,fontsize=8,color=NESA_COLORS['muted'],va='top',ha='left')

def limpar_eixos(ax, manter_esquerda=True):
    for lado in ['top','right']:
        ax.spines[lado].set_visible(False)
    if not manter_esquerda:
        ax.spines['left'].set_visible(False)
    ax.grid(axis='x', visible=False)
    ax.tick_params(length=0)

# --- VMP e unidades lidos do YAML normativo (fonte unica) ---
def carregar_norma(yaml_path):
    norma = yaml.safe_load(open(yaml_path, encoding='utf-8'))
    vmp, unidades, rotulos = {}, {}, {}
    for p, d in norma['parametros'].items():
        vmp[p] = (d['minimo'], d['maximo']) if d['tipo'] == 'faixa' else d['maximo']
        unidades[p] = d.get('unidade', 'mg/L')
        rotulos[p] = d.get('rotulo', p.replace('_', ' ').title())
    return vmp, unidades, rotulos, norma

VMP_CONAMA, UNIDADES, ROTULOS, NORMA = carregar_norma(FILE_NORMA)
print(f"[*] Limites carregados de: {FILE_NORMA.name}  ({len(VMP_CONAMA)} parametros)  | norma {NORMA['norma']['id']}")

def nome_exibicao(param):
    return ROTULOS.get(param, param.replace('_', ' ').title())

def fmt_num(x):
    if not np.isfinite(x):
        return "-"
    a = abs(x)
    if a == 0: return "0"
    if a >= 100: return f"{x:,.0f}"
    if a >= 1: return f"{x:.1f}"
    if a >= 0.01: return f"{x:.3f}"
    return f"{x:.2g}"

def contar_outliers_iqr(serie):
    q1, q3 = np.percentile(serie, [25, 75]); iqr = q3 - q1
    if iqr == 0: return 0
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((serie < lo) | (serie > hi)).sum())

def detectar_rupturas(datas, valores, min_size=3):
    if not _HAS_RUPT: return 0, []
    vals = np.asarray(valores, dtype=float); n = len(vals)
    if n < 2 * min_size or np.nanstd(vals) == 0: return 0, []
    sinal = (vals - np.nanmean(vals)) / (np.nanstd(vals) + 1e-9)
    try:
        bkps = rpt.Pelt(model='l2', min_size=min_size, jump=1).fit(sinal).predict(pen=2*np.log(n))
        idxs = [b for b in bkps if 0 < b < n]
        return len(idxs), [pd.Timestamp(datas.iloc[i]).strftime('%m/%Y') for i in idxs]
    except Exception:
        return 0, []

def posthoc_dunn_param(df, parametro, col_ete):
    if not _HAS_DUNN: return []
    sub = df[[col_ete, parametro]].dropna()
    validas = [e for e in sub[col_ete].unique() if len(sub[sub[col_ete]==e])>=3 and sub[sub[col_ete]==e][parametro].std()>0]
    sub = sub[sub[col_ete].isin(validas)]
    if sub[col_ete].nunique() < 2: return []
    try:
        m = sp.posthoc_dunn(sub, val_col=parametro, group_col=col_ete, p_adjust='holm')
    except Exception:
        return []
    etes = list(m.columns); res = []
    for i in range(len(etes)):
        for j in range(i+1, len(etes)):
            p = float(m.loc[etes[i], etes[j]])
            res.append({'Parâmetro':nome_exibicao(parametro),'ETE A':etes[i],'ETE B':etes[j],
                        'p-value (Dunn/Holm)':round(p,4),'Diferem?':'SIM' if p<0.05 else 'NÃO'})
    return res

def validar_schema(df, col_ete, parametros):
    if not _HAS_PANDERA:
        return None, pd.DataFrame([{'Verificacao':'pandera ausente','Resultado':'PULADO'}])
    cols = {col_ete: pa.Column(str, nullable=False)}
    if 'DATA' in df.columns:
        cols['DATA'] = pa.Column('datetime64[ns]', nullable=True, coerce=True)
    for p in parametros:
        cols[p] = pa.Column(float, nullable=True, coerce=True, checks=pa.Check.ge(-1e-9))
    try:
        pa.DataFrameSchema(cols, strict=False).validate(df, lazy=True)
        return True, pd.DataFrame([{'Verificacao':'Schema (tipos+dominios)','Resultado':'CONFORME',
                                    'Detalhe':f'{len(parametros)} parametros, {len(df)} linhas'}])
    except pa.errors.SchemaErrors as err:
        fc = err.failure_cases[['column','check','failure_case']].rename(
            columns={'column':'Coluna','check':'Regra','failure_case':'Valor'})
        fc.insert(0,'Resultado','NAO CONFORME')
        return False, fc

def analisar_parametro(df, parametro, limite_conama, col_ete, rupturas_detalhe):
    resultados = []
    for ete in df[col_ete].dropna().unique():
        df_ete = df[df[col_ete] == ete].sort_values('DATA')
        serie = df_ete[parametro].dropna()
        if len(serie) < 3: continue
        media, desvio = serie.mean(), serie.std()
        minimo, maximo = serie.min(), serie.max()
        p95 = np.percentile(serie, 95)
        if isinstance(limite_conama, tuple):
            vmin, vmax = limite_conama
            amostras_ok = len(serie[(serie >= vmin) & (serie <= vmax)])
            status_auditoria = "ROBUSTO" if (serie.min() >= vmin and p95 <= vmax) else "RISCO"
            limite_str = f"{vmin} a {vmax}"
        else:
            amostras_ok = len(serie[serie <= limite_conama])
            status_auditoria = "ROBUSTO" if p95 <= limite_conama else "RISCO"
            limite_str = str(limite_conama)
        taxa = (amostras_ok / len(serie)) * 100
        if desvio > 0 and len(serie) >= 4:
            try:
                mkr = mk.original_test(serie); tendencia, pmk = mkr.trend, mkr.p
                tendencia = {'increasing':'Crescente (Piora)','decreasing':'Decrescente (Melhora)'}.get(tendencia,'Estável')
            except Exception:
                tendencia, pmk = "Erro Cálculo", np.nan
        else:
            tendencia, pmk = "Sem variância/dados", np.nan
        n_out = contar_outliers_iqr(serie)
        datas_serie = df_ete.loc[serie.index, 'DATA'].reset_index(drop=True)
        n_rup, datas_rup = detectar_rupturas(datas_serie, serie.values)
        if n_rup > 0:
            rupturas_detalhe.append({'Parâmetro':nome_exibicao(parametro),'ETE':ete,
                                     'Rupturas (n)':n_rup,'Datas das Rupturas':", ".join(datas_rup)})
        resultados.append({'Parâmetro':nome_exibicao(parametro),'ETE':ete,'N':len(serie),
            'Mínimo':round(minimo,3),'Média':round(media,3),'Máximo':round(maximo,3),
            'Desvio Padrão':round(desvio,3),'Outliers (IQR)':n_out,'P95 (Auditoria)':round(p95,3),
            'VMP CONAMA':limite_str,'Conformidade P95 (%)':f"{taxa:.1f}%",'Avaliação (P95)':status_auditoria,
            'Tendência (MK)':tendencia,'p-value (MK)':round(pmk,4) if pd.notna(pmk) else "N/A",'Rupturas (n)':n_rup})
    return resultados

def teste_kruskal_wallis_geral(df, parametro, col_ete):
    grupos, nomes = [], []
    for ete in df[col_ete].dropna().unique():
        serie = df[df[col_ete]==ete][parametro].dropna()
        if len(serie) >= 3 and serie.std() > 0:
            grupos.append(serie); nomes.append(ete)
    if len(grupos) > 1:
        stat, pv = stats.kruskal(*grupos)
        return {'Parâmetro':nome_exibicao(parametro),'ETEs Comparadas':" vs ".join(nomes),
                'Estatística (H)':round(stat,3),'p-value (KW)':round(pv,4),
                'Diferença Significativa?':"SIM" if pv<0.05 else "NÃO"}, (pv < 0.05)
    return None, False

def plotar_serie_temporal(df, parametro, ete, col_ete, dir_out):
    df_plot = df[df[col_ete]==ete].sort_values('DATA').dropna(subset=[parametro])
    if len(df_plot) < 2: return
    serie = df_plot[parametro]; limite = VMP_CONAMA[parametro]
    p95 = np.percentile(serie, 95); unidade = UNIDADES.get(parametro, 'mg/L')
    fig, ax = plt.subplots(figsize=(11, 5.2)); limpar_eixos(ax); ytrans = ax.get_yaxis_transform()
    if isinstance(limite, tuple):
        ax.axhspan(limite[0], limite[1], color=NESA_COLORS['ok'], alpha=0.06, zorder=0)
        for y in limite:
            ax.axhline(y, color=NESA_COLORS['vmp'], linestyle='--', linewidth=1.4, zorder=1)
            ax.text(1.005, y, f' VMP {y:g}', transform=ytrans, color=NESA_COLORS['vmp'], fontsize=9, va='center', fontweight='bold')
        topo, base = limite[1], limite[0]
    else:
        ax.axhline(limite, color=NESA_COLORS['vmp'], linestyle='--', linewidth=1.6, zorder=1)
        ax.text(1.005, limite, f' VMP {limite:g}', transform=ytrans, color=NESA_COLORS['vmp'], fontsize=9, va='center', fontweight='bold')
        topo, base = limite, 0.0
    ax.plot(df_plot['DATA'], serie, marker='o', markersize=5, linewidth=2.4, color=NESA_COLORS['primary'],
            zorder=3, markerfacecolor='white', markeredgecolor=NESA_COLORS['primary'], markeredgewidth=1.6)
    ax.axhline(p95, color=NESA_COLORS['secondary'], linestyle=':', linewidth=1.8, zorder=2)
    ax.text(1.005, p95, f' P95 {fmt_num(p95)}', transform=ytrans, color=NESA_COLORS['secondary'], fontsize=9, va='center')
    datas_serie = df_plot['DATA'].reset_index(drop=True)
    n_rup, datas_rup = detectar_rupturas(datas_serie, serie.values)
    for k, ts in enumerate(datas_rup):
        x = pd.to_datetime(ts, format='%m/%Y')
        ax.axvline(x, color=NESA_COLORS['muted'], linestyle='--', linewidth=1.1, alpha=0.8, zorder=1)
        if k == 0:
            ax.text(x, 1.005, ' ruptura', transform=ax.get_xaxis_transform(), color=NESA_COLORS['muted'], fontsize=8, va='bottom', ha='left')
    x_ult, y_ult = df_plot['DATA'].iloc[-1], serie.iloc[-1]
    ax.annotate(f'{fmt_num(y_ult)}', xy=(x_ult, y_ult), xytext=(8,0), textcoords='offset points',
                va='center', fontsize=9.5, fontweight='bold', color=NESA_COLORS['primary'])
    y_max = max(topo, float(serie.max())) * 1.12
    y_min = min(float(serie.min()), base) * 0.95 if isinstance(limite, tuple) else min(0.0, float(serie.min()))
    ax.set_ylim(y_min, y_max)
    conforme = (serie.between(limite[0], limite[1]).mean() if isinstance(limite, tuple) else (serie <= limite).mean()) * 100
    rotulo_y = "pH" if unidade == '' else f"Valor ({unidade})"
    extra = f"  |  {n_rup} ruptura(s) de regime" if n_rup else ""
    sub = (f"{conforme:.0f}% das {len(serie)} amostras dentro do limite  |  P95 = {fmt_num(p95)} {unidade}{extra}").strip()
    titulo_storytelling(ax, f"{nome_exibicao(parametro)}  -  {ete}", sub,
        "Fonte: Monitoramento NESA   |   VMP: CONAMA 430/11 (Art. 21 / Tabela I), extraido do PDF oficial — conformidade amostral no Relatorio v4")
    ax.set_ylabel(rotulo_y); formatar_meses_pt(ax); plt.setp(ax.get_xticklabels(), rotation=0, ha='center')
    fig.savefig(dir_out / f"Plot_{parametro}_{ete.replace(' ', '_')}.png"); plt.close(fig)

if __name__ == "__main__":
    try:
        print(f"[*] Camadas -> Dunn:{_HAS_DUNN} Ruptures:{_HAS_RUPT} Pandera:{_HAS_PANDERA}")
        df = pd.read_excel(FILE_INPUT)
        col_ete = [c for c in df.columns if 'DESCRICAO' in c.upper() and 'PONTO' in c.upper()][0]
        df[col_ete] = df[col_ete].astype(str).str.upper().str.strip().replace({'ETE 1':'ETE 01','ETE 2':'ETE 02'})
        presentes = [p for p in VMP_CONAMA if p in df.columns]
        ok_schema, df_val = validar_schema(df, col_ete, presentes)
        print(f"[*] Validacao schema (pandera): {'CONFORME' if ok_schema else ('NAO CONFORME' if ok_schema is False else 'PULADO')}")
        rel, rel_kw, rel_dunn, rupturas = [], [], [], []
        print("[*] Processando estatistica (v4)...")
        for param in presentes:
            rel.extend(analisar_parametro(df, param, VMP_CONAMA[param], col_ete, rupturas))
            res_kw, signif = teste_kruskal_wallis_geral(df, param, col_ete)
            if res_kw:
                rel_kw.append(res_kw)
                if signif: rel_dunn.extend(posthoc_dunn_param(df, param, col_ete))
            for ete in df[col_ete].unique():
                plotar_serie_temporal(df, param, ete, col_ete, DIR_PRODUTOS)
        caminho = DIR_PRODUTOS / "Relatorio_Analitico_Mestre_v4.xlsx"
        with pd.ExcelWriter(caminho, engine='openpyxl') as w:
            pd.DataFrame(rel).to_excel(w, "Estatistica_e_Tendencia", index=False)
            if rel_kw: pd.DataFrame(rel_kw).to_excel(w, "Kruskal_Wallis", index=False)
            if rel_dunn: pd.DataFrame(rel_dunn).to_excel(w, "PostHoc_Dunn", index=False)
            if rupturas: pd.DataFrame(rupturas).to_excel(w, "Pontos_de_Ruptura", index=False)
            df_val.to_excel(w, "Validacao_Schema", index=False)
        print(f"\n[+] Relatorio Analitico v4: {caminho.name}")
    except Exception as e:
        print(f"[!] Erro critico: {e}")


[*] Limites carregados de: conama_430_2011.yaml  (36 parametros)  | norma CONAMA-430-2011
[*] Camadas -> Dunn:True Ruptures:True Pandera:True


[*] Validacao schema (pandera): CONFORME
[*] Processando estatistica (v4)...



[+] Relatorio Analitico v4: Relatorio_Analitico_Mestre_v4.xlsx


C:\Users\thiago\AppData\Local\Temp\ipykernel_33844\506143918.py:288: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  pd.DataFrame(rel).to_excel(w, "Estatistica_e_Tendencia", index=False)
C:\Users\thiago\AppData\Local\Temp\ipykernel_33844\506143918.py:289: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  if rel_kw: pd.DataFrame(rel_kw).to_excel(w, "Kruskal_Wallis", index=False)
C:\Users\thiago\AppData\Local\Temp\ipykernel_33844\506143918.py:290: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  if rel_dunn: pd.DataFrame(rel_dunn).to_excel(w, "PostHoc_Dunn", index=False)
C:\Users\thiago\AppData\Local\Temp\ipykernel_33844\506143918.py:291: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the ar

In [2]:
# =============================================================================
# v4 (06-06-2026) — Modulo QA/QC: tabela + mapa de calor dos dados <LQ.
# =============================================================================
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore", message="invalid value encountered")

RAIZ = Path(r"C:\Users\thiago\Documents\estudos_python\etes_nesa")
DIR_BASE = RAIZ / "base_dados"
DIR_PRODUTOS = RAIZ / "Efluentes_Sanitários" / "produtos"
FILE_INPUT = DIR_BASE / "ETEs_NorteEnergia_Processado_v1.xlsx"
DIR_PRODUTOS.mkdir(parents=True, exist_ok=True)
NESA_COLORS = {'primary':'#009CA7','text':'#404041','muted':'#9aa0a6'}
plt.rcParams.update({'font.family':'sans-serif','font.sans-serif':['Segoe UI','Calibri','Arial','DejaVu Sans'],
                     'savefig.dpi':300,'savefig.bbox':'tight'})
try:
    print("[*] QA/QC: lendo base processada...")
    df = pd.read_excel(FILE_INPUT)
    col_ete = [c for c in df.columns if 'DESCRICAO' in c.upper() and 'PONTO' in c.upper()][0]
    df[col_ete] = df[col_ete].astype(str).str.upper().str.strip().replace({'ETE 1':'ETE 01','ETE 2':'ETE 02'})
    meta = df.columns[:4].tolist()
    parametros = [c for c in df.columns if c not in meta and not c.endswith('_IMPUTADO_LQ') and not c.endswith('_STATUS_VMP')]
    dados = []
    for param in parametros:
        flag = param + "_IMPUTADO_LQ"; has = flag in df.columns
        for ete in df[col_ete].unique():
            de = df[df[col_ete]==ete]; s = de[param].dropna(); n = len(s)
            if n > 0:
                nc = de.loc[s.index, flag].fillna(False).sum() if has else 0
                dados.append({'ETE':ete,'Parâmetro':param.replace('_',' '),'Total de Amostras (N)':int(n),
                              'Dados Censurados (<LQ)':int(nc),'% Censurado':round(nc/n*100,1)})
    df_qc = pd.DataFrame(dados)
    df_qc.to_excel(DIR_PRODUTOS / "Relatorio_QAQC_Dados_Censurados.xlsx", index=False)
    dff = df_qc[df_qc['% Censurado'] > 0]
    if not dff.empty:
        piv = dff.pivot(index='Parâmetro', columns='ETE', values='% Censurado').fillna(0)
        piv = piv.loc[piv.mean(axis=1).sort_values(ascending=False).index]
        fig, ax = plt.subplots(figsize=(9, max(6, len(piv)*0.42)))
        sns.heatmap(piv, annot=True, fmt='.0f', cmap='YlGnBu', vmin=0, vmax=100, linewidths=1.2, linecolor='white',
                    annot_kws={'fontsize':9,'fontweight':'bold'}, cbar_kws={'label':'% de amostras censuradas (<LQ)','shrink':0.55}, ax=ax)
        ax.set_xlabel(''); ax.set_ylabel(''); ax.tick_params(length=0); plt.setp(ax.get_xticklabels(), fontweight='bold')
        ax.text(0.0,1.06,"Matriz de Dados Censurados (<LQ) por ETE",transform=ax.transAxes,fontsize=14,fontweight='bold',color='#2b2b2b',va='bottom',ha='left')
        ax.text(0.0,1.015,"Células mais escuras = maior fração de amostras abaixo do limite de quantificação (não detectado)",transform=ax.transAxes,fontsize=9.5,color=NESA_COLORS['muted'],va='bottom',ha='left')
        fig.savefig(DIR_PRODUTOS / "Heatmap_QAQC_Censurados.png"); plt.close(fig)
    print("[+] QA/QC concluido.")
    print(df_qc.sort_values('% Censurado', ascending=False).head(10).to_string(index=False))
except Exception as e:
    print(f"[!] Erro QA/QC: {e}")


[*] QA/QC: lendo base processada...


[+] QA/QC concluido.
         ETE                                Parâmetro  Total de Amostras (N)  Dados Censurados (<LQ)  % Censurado
      ETE 01                            ARSENIO TOTAL                     26                      26        100.0
      ETE PM                            ARSENIO TOTAL                     20                      20        100.0
ETE COMPACTA                         CROMO TRIVALENTE                     24                      24        100.0
      ETE PM                         CROMO TRIVALENTE                     20                      20        100.0
      ETE 01 DICLOROETENO SOMATORIA DE 1112CIS12TRANS                     26                      26        100.0
      ETE 02                           MERCURIO TOTAL                     26                      26        100.0
      ETE 02                              ETILBENZENO                     26                      26        100.0
      ETE 01                              ETILBENZENO              

In [3]:
# =============================================================================
# v4 (06-06-2026) — Dossie Visual Segregado por UHE. P95 SEGMENTADO.
# =============================================================================
import datetime, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
warnings.filterwarnings("ignore", message="invalid value encountered")
warnings.filterwarnings("ignore", message="Mean of empty slice")

RAIZ = Path(r"C:\Users\thiago\Documents\estudos_python\etes_nesa")
DIR_BASE = RAIZ / "base_dados"
DIR_PRODUTOS = RAIZ / "Efluentes_Sanitários" / "produtos"
FILE_VAZAO = DIR_BASE / "vazão turbinada.xlsx"
FILE_INPUT = DIR_BASE / "ETEs_NorteEnergia_Processado_v1.xlsx"
DIR_PRODUTOS.mkdir(parents=True, exist_ok=True)
NESA_COLORS = {'rio':'#B9F8FF','ete_barras':'#009CA7','linha_vmp':'#F25A3C','dbo':'#009CA7','nh3':'#52BD8B','fosforo':'#FEBD2A','lab':'#404041','muted':'#9aa0a6'}
plt.rcParams.update({'font.family':'sans-serif','font.sans-serif':['Segoe UI','Calibri','Arial','DejaVu Sans'],'axes.axisbelow':True,'savefig.dpi':600,'savefig.bbox':'tight'})
MESES_PT = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
def formatar_meses_pt(ax):
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,pos=None: f"{MESES_PT[mdates.num2date(x).month-1]}/{mdates.num2date(x):%y}"))
def titulo_storytelling(ax, titulo, subtitulo=None, fonte=None):
    ax.set_title('')
    ax.text(0.0,1.12,titulo,transform=ax.transAxes,fontsize=15,fontweight='bold',color='#2b2b2b',va='bottom',ha='left')
    if subtitulo: ax.text(0.0,1.035,subtitulo,transform=ax.transAxes,fontsize=10.5,color=NESA_COLORS['muted'],va='bottom',ha='left')
    if fonte: ax.text(0.0,-0.32,fonte,transform=ax.transAxes,fontsize=8,color=NESA_COLORS['muted'],va='top',ha='left')
def limpar_eixos(ax):
    for lado in ['top','right']: ax.spines[lado].set_visible(False)
    ax.tick_params(length=0)
def extrair_data_segura(val):
    if pd.isna(val): return pd.NaT
    if isinstance(val,(pd.Timestamp,datetime.datetime)): return pd.Timestamp(year=val.year,month=val.month,day=1)
    s=str(val).lower(); meses=['jan','fev','mar','abr','mai','jun','jul','ago','set','out','nov','dez']
    for i,mes in enumerate(meses):
        if mes in s:
            anos=re.findall(r'\d{2,4}',s)
            if anos:
                ano=int(anos[-1]); ano+=2000 if ano<100 else 0
                return pd.Timestamp(year=ano,month=i+1,day=1)
    return pd.NaT
def calcular_p95_segmento(df, col_ete, etes, candidatos, fallback):
    dseg = df if col_ete is None else df[df[col_ete].isin(etes)]
    for nome in candidatos:
        if nome in dseg.columns:
            serie = pd.to_numeric(dseg[nome], errors='coerce').dropna()
            if len(serie) >= 3:
                return round(float(np.percentile(serie,95)),2), f"P95 de '{nome}' em {etes} (N={len(serie)})"
    return fallback, f"FALLBACK (sem dados p/ {candidatos} em {etes})"
PARAM_COLS = {'DBO':['DEMANDA_BIOQUIMICA_DE_OXIGENIO'],'NH3':['NITROGENIO_AMONIACAL_TOTAL','NITROGENIO_AMONIACAL','N_AMONIACAL','NITROGENIO_AMONIACAL_NH3'],'FOSF':['FOSFORO_TOTAL','FOSFORO','P_TOTAL','FOSFORO_TOTAL_P']}
FALLBACKS = {'DBO':88.78,'NH3':45.0,'FOSF':4.5}
SEGMENTOS = {'BM':['ETE 01','ETE 02'],'PM':['ETE PM','ETE COMPACTA']}
dados_gri = {
    'ETE_01':[651.4,635.7,503.4,675.4,639.4,569.0,566.6,547.0,455.1,679.9,581.8,551.6,324.8,169.6,99.2,155.2,129.6,110.4,137.6,120.0,146.4,160.0,144.8,92.0,107.2,111.2,145.6],
    'ETE_02':[597.4,525.1,415.0,535.8,579.8,469.4,534.2,438.6,330.9,581.3,405.8,387.2,2414.4,1060.8,959.2,1325.6,1130.4,640.8,885.6,580.0,664.8,688.0,585.6,648.8,602.4,502.4,623.76],
    'ETE_PM':[87.0,198.4,206.1,185.6,215.0,245.1,294.4,258.4,291.0,338.5,308.6,314.5,281.6,146.4,133.76,137.92,142.56,141.12,126.72,104.16,104.48,0.0,109.92,0.0,0.0,57.6,65.92],
    'ETE_COMPACTA':[21.8,49.6,51.5,46.4,50.6,52.5,52.8,52.0,45.4,52.3,46.2,46.7,41.6,16.0,18.24,22.08,23.04,20.48,18.88,15.84,19.76,25.12,19.68,19.04,14.72,14.4,16.48]}
df_gri = pd.DataFrame(dados_gri); df_gri['DATA_DT'] = pd.date_range(start='2024-01-01', periods=len(df_gri), freq='MS')
LIMITE_INICIO = pd.Timestamp('2023-12-15'); LIMITE_FIM = pd.Timestamp('2026-03-15')
try:
    p95, origem = {}, {}
    if FILE_INPUT.exists():
        dfp = pd.read_excel(FILE_INPUT)
        col_ete = next((c for c in dfp.columns if 'DESCRICAO' in c.upper() and 'PONTO' in c.upper()), None)
        if col_ete is not None:
            dfp[col_ete] = dfp[col_ete].astype(str).str.upper().str.strip().replace({'ETE 1':'ETE 01','ETE 2':'ETE 02'})
        for p, cand in PARAM_COLS.items():
            for seg, etes in SEGMENTOS.items():
                p95[(p,seg)], origem[(p,seg)] = calcular_p95_segmento(dfp, col_ete, etes, cand, FALLBACKS[p])
    else:
        for p in PARAM_COLS:
            for seg in SEGMENTOS: p95[(p,seg)], origem[(p,seg)] = FALLBACKS[p], "FALLBACK (base nao encontrada)"
    print("[*] P95 SEGMENTADO por UHE:")
    for seg in SEGMENTOS:
        print(f"  --- {seg} ({' + '.join(SEGMENTOS[seg])}) ---")
        for p in PARAM_COLS: print(f"      {p:5s} = {p95[(p,seg)]:8.2f} mg/L  ->  {origem[(p,seg)]}")
    dfv = pd.read_excel(FILE_VAZAO); dfv.columns = ['Mes_Original','Q_Pimental_m3s','Q_BeloMonte_m3s']
    dfv = dfv.dropna(subset=['Mes_Original']); dfv['DATA_DT'] = dfv['Mes_Original'].apply(extrair_data_segura)
    dfv = dfv.dropna(subset=['DATA_DT'])
    df_final = pd.merge(df_gri, dfv, on='DATA_DT', how='left')
    for seg, q_col, etes in [('BM','Q_BeloMonte_m3s',['ETE_01','ETE_02']),('PM','Q_Pimental_m3s',['ETE_PM','ETE_COMPACTA'])]:
        df_final[f'Vol_Rio_{seg}_m3mes'] = df_final[q_col]*86400*30.44
        df_final[f'Vol_Efluente_{seg}'] = df_final[etes].sum(axis=1)
        df_final[f'Fator_Diluicao_{seg}'] = df_final[f'Vol_Rio_{seg}_m3mes']/df_final[f'Vol_Efluente_{seg}']
        for nome in ['DBO','NH3','FOSF']:
            df_final[f'Adicao_{nome}_{seg}_mgL'] = (df_final[f'Vol_Efluente_{seg}']*p95[(nome,seg)])/df_final[f'Vol_Rio_{seg}_m3mes']
    def gerar_grafico_diluicao(fator_col, uhe, arq):
        fig, ax = plt.subplots(figsize=(14,6)); limpar_eixos(ax)
        fm = df_final[fator_col]/1e6; idx_min = fm.idxmin() if fm.notna().any() else None
        cores = [NESA_COLORS['linha_vmp'] if i==idx_min else NESA_COLORS['ete_barras'] for i in fm.index]
        bars = ax.bar(df_final['DATA_DT'], fm, width=22, color=cores, alpha=0.92)
        ax.bar_label(bars, labels=[f'{v:.1f}' if pd.notna(v) else '' for v in fm], padding=4, fontsize=8.5, color='#404041')
        ax.set_ylim(0, fm.max()*1.18 if pd.notna(fm.max()) else 10); ax.set_xlim(LIMITE_INICIO, LIMITE_FIM)
        ax.grid(axis='y', linestyle='--', alpha=0.35); ax.grid(axis='x', visible=False)
        mn = fm.min(); subt = ""
        if pd.notna(mn):
            xm = df_final['DATA_DT'].iloc[df_final.index.get_loc(idx_min)]
            ax.annotate(f'Pior mês: 1 p/ {mn:.1f} M', xy=(xm,mn), xytext=(0,28), textcoords='offset points', ha='center',
                        fontsize=9.5, fontweight='bold', color=NESA_COLORS['linha_vmp'], arrowprops=dict(arrowstyle='->', color=NESA_COLORS['linha_vmp'], lw=1.6))
            subt = f"Mesmo no pior mês histórico, cada 1 L de efluente é diluído em {mn:.1f} milhões de litros do Rio Xingu"
        titulo_storytelling(ax, f"Proporção Hidrodinâmica - UHE {uhe}", subt, "Fonte: Volumes GRI 303-4 (NESA) x Vazão turbinada (ONS)   |   Vol.rio = Q x 86400 x 30,44")
        ax.set_ylabel("Milhões de litros de rio por 1 L de efluente", fontweight='bold')
        formatar_meses_pt(ax); plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
        fig.savefig(DIR_PRODUTOS / arq); plt.close(fig)
    def gerar_grafico_impacto(col, cor, nome, p95_seg, lab, nota, ylim, uhe, arq):
        fig, ax = plt.subplots(figsize=(12,5.4)); limpar_eixos(ax)
        m = df_final[col].notna()
        ax.plot(df_final['DATA_DT'], df_final[col], color=cor, linewidth=3, label=f'Adição efetiva de {nome}', zorder=3)
        ax.fill_between(df_final['DATA_DT'][m], 0, df_final[col][m], color=cor, alpha=0.18)
        ax.axhline(lab, color=NESA_COLORS['lab'], linestyle='-', linewidth=2, label=f'Limite de detecção lab. ({lab} mg/L)')
        ax.set_ylim(0, ylim); ax.set_xlim(LIMITE_INICIO, LIMITE_FIM)
        ax.annotate(nota, xy=(pd.Timestamp('2025-01-01'), ylim*0.96), xytext=(pd.Timestamp('2025-01-01'), ylim*0.82),
                    color=NESA_COLORS['linha_vmp'], fontweight='bold', fontsize=10.5, ha='center', arrowprops=dict(facecolor=NESA_COLORS['linha_vmp'], arrowstyle='->', lw=2))
        am = df_final[col].max()
        subt = (f"P95 do efluente = {p95_seg:g} mg/L -> adição máxima de {am:.5f} mg/L  ({lab/am:,.0f}x abaixo do limite de detecção do laboratório)") if (pd.notna(am) and am>0) else f"P95 do efluente = {p95_seg:g} mg/L -> adição próxima de zero após diluição"
        titulo_storytelling(ax, f"Modelagem de Risco - UHE {uhe}: {nome}", subt, "Fonte: Balanço de massa NESA (P95 segmentado x diluição do Rio Xingu)   |   VMP CONAMA 430/11 (Art. 21)")
        ax.set_ylabel("Concentração adicionada (mg/L)", fontweight='bold')
        ax.grid(axis='y', linestyle=':', alpha=0.5); ax.grid(axis='x', visible=False)
        formatar_meses_pt(ax); ax.legend(loc='upper center', bbox_to_anchor=(0.5,-0.16), ncol=2); plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
        fig.savefig(DIR_PRODUTOS / arq); plt.close(fig)
    gerar_grafico_diluicao('Fator_Diluicao_BM','Belo Monte',"Produto1A_Diluicao_BeloMonte.png")
    gerar_grafico_impacto('Adicao_DBO_BM_mgL',NESA_COLORS['dbo'],'DBO',p95[('DBO','BM')],2.0,'Limite CONAMA 120 mg/L (Art. 21) -> Fora de escala',2.5,'Belo Monte',"Produto2A_Impacto_DBO_BM.png")
    gerar_grafico_impacto('Adicao_NH3_BM_mgL',NESA_COLORS['nh3'],'N. Amoniacal',p95[('NH3','BM')],0.5,'Nao exigivel p/ esgoto sanitario (Art. 21 §1) -> Fora de escala',0.7,'Belo Monte',"Produto3A_Impacto_NH3_BM.png")
    gerar_grafico_impacto('Adicao_FOSF_BM_mgL',NESA_COLORS['fosforo'],'Fósforo Total',p95[('FOSF','BM')],0.5,'Risco de eutrofização -> Fora de escala',0.7,'Belo Monte',"Produto4A_Impacto_Fosforo_BM.png")
    gerar_grafico_diluicao('Fator_Diluicao_PM','Pimental (TVR)',"Produto1B_Diluicao_Pimental.png")
    gerar_grafico_impacto('Adicao_DBO_PM_mgL',NESA_COLORS['dbo'],'DBO',p95[('DBO','PM')],2.0,'Limite CONAMA 120 mg/L (Art. 21) -> Fora de escala',2.5,'Pimental',"Produto2B_Impacto_DBO_PM.png")
    gerar_grafico_impacto('Adicao_NH3_PM_mgL',NESA_COLORS['nh3'],'N. Amoniacal',p95[('NH3','PM')],0.5,'Nao exigivel p/ esgoto sanitario (Art. 21 §1) -> Fora de escala',0.7,'Pimental',"Produto3B_Impacto_NH3_PM.png")
    gerar_grafico_impacto('Adicao_FOSF_PM_mgL',NESA_COLORS['fosforo'],'Fósforo Total',p95[('FOSF','PM')],0.5,'Risco de eutrofização -> Fora de escala',0.7,'Pimental',"Produto4B_Impacto_Fosforo_PM.png")
    df_final['Mes/Ano'] = df_final['DATA_DT'].dt.strftime('%m/%Y')
    df_final['P95_DBO_BM'] = p95[('DBO','BM')]; df_final['P95_DBO_PM'] = p95[('DBO','PM')]
    cols = ['Mes/Ano','Q_BeloMonte_m3s','Vol_Rio_BM_m3mes','Vol_Efluente_BM','Fator_Diluicao_BM','Adicao_DBO_BM_mgL','Adicao_NH3_BM_mgL','Adicao_FOSF_BM_mgL','Q_Pimental_m3s','Vol_Rio_PM_m3mes','Vol_Efluente_PM','Fator_Diluicao_PM','Adicao_DBO_PM_mgL','Adicao_NH3_PM_mgL','Adicao_FOSF_PM_mgL']
    df_final[cols].to_excel(DIR_PRODUTOS / "Relatorio_Diluicao_Mestre_Segregado.xlsx", index=False)
    print("\n[*] Dossie hidrodinamico concluido (8 graficos).")
except Exception as e:
    print(f"\n[!] ERRO hidrodinamico: {e}")


[*] P95 SEGMENTADO por UHE:
  --- BM (ETE 01 + ETE 02) ---
      DBO   =    86.12 mg/L  ->  P95 de 'DEMANDA_BIOQUIMICA_DE_OXIGENIO' em ['ETE 01', 'ETE 02'] (N=78)
      NH3   =   266.05 mg/L  ->  P95 de 'NITROGENIO_AMONIACAL' em ['ETE 01', 'ETE 02'] (N=78)
      FOSF  =     8.80 mg/L  ->  P95 de 'FOSFORO_TOTAL' em ['ETE 01', 'ETE 02'] (N=78)
  --- PM (ETE PM + ETE COMPACTA) ---
      DBO   =   188.37 mg/L  ->  P95 de 'DEMANDA_BIOQUIMICA_DE_OXIGENIO' em ['ETE PM', 'ETE COMPACTA'] (N=85)
      NH3   =   184.16 mg/L  ->  P95 de 'NITROGENIO_AMONIACAL' em ['ETE PM', 'ETE COMPACTA'] (N=85)
      FOSF  =    27.83 mg/L  ->  P95 de 'FOSFORO_TOTAL' em ['ETE PM', 'ETE COMPACTA'] (N=85)



[*] Dossie hidrodinamico concluido (8 graficos).


In [4]:
# =============================================================================
# Autor:   Thiago Souza (@engthiago1979-blip)  |  v4 (06-06-2026)
# Skill:   analista-python-ambiental
# Uso:     VERIFICACAO AUTOMATIZADA dos limites CONAMA 430/11 a partir do YAML
#          normativo (Art. 21-23 + Tabela I). Conformidade AMOSTRAL (amostra a
#          amostra, conforme Art. 24), respeitando a aplicabilidade:
#            - obrigatorio_art21   (exigivel)
#            - condicional_tabela1 (a criterio do orgao - Art. 21 §1)
#            - nao_exigivel        (ex.: N-amoniacal p/ esgoto sanitario)
#          Rastreabilidade: verifica o sha256 do PDF fonte.
# =============================================================================
import hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore", message="invalid value encountered")

RAIZ = Path(r"C:\Users\thiago\Documents\estudos_python\etes_nesa")
DIR_BASE = RAIZ / "base_dados"
DIR_PROJ = RAIZ / "Efluentes_Sanitários"
DIR_PRODUTOS = DIR_PROJ / "produtos"
FILE_INPUT = DIR_BASE / "ETEs_NorteEnergia_Processado_v1.xlsx"
FILE_NORMA = DIR_PROJ / "normas" / "conama_430_2011.yaml"
DIR_PRODUTOS.mkdir(parents=True, exist_ok=True)
NESA_COLORS = {'text':'#404041','muted':'#9aa0a6'}
plt.rcParams.update({'font.family':'sans-serif','font.sans-serif':['Segoe UI','Calibri','Arial','DejaVu Sans'],'savefig.dpi':300,'savefig.bbox':'tight'})

CAT_LABEL = {'obrigatorio_art21':'Obrigatório (Art. 21)','condicional_tabela1':'Condicional (Tabela I / Art. 21 §1)','nao_exigivel':'Não exigível (Art. 21 §1)'}

def sha256_arquivo(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for bloco in iter(lambda: f.read(8192), b''):
            h.update(bloco)
    return h.hexdigest()

def conforme_amostra(valor, par):
    if par['tipo'] == 'faixa':
        return par['minimo'] <= valor <= par['maximo']
    return valor <= par['maximo']

def limite_str(par):
    return f"{par['minimo']:g} a {par['maximo']:g}" if par['tipo'] == 'faixa' else f"<= {par['maximo']:g}"

try:
    norma = yaml.safe_load(open(FILE_NORMA, encoding='utf-8'))
    meta, params = norma['norma'], norma['parametros']

    # --- Integridade da fonte (sha256) ---
    pdf_path = RAIZ / meta['fonte_arquivo']
    if pdf_path.exists():
        sha_atual = sha256_arquivo(pdf_path)
        integridade = "OK (hash confere)" if sha_atual == meta['sha256_fonte'] else "ALERTA: hash divergente!"
    else:
        sha_atual, integridade = "-", "PDF fonte nao encontrado"
    print(f"[*] Norma {meta['id']} | fonte: {meta['fonte_arquivo']} | integridade: {integridade}")
    print(f"[*] Status: {meta['status']} | revisado_por: {meta['revisado_por']} (revisao tecnica pendente)")

    df = pd.read_excel(FILE_INPUT)
    col_ete = [c for c in df.columns if 'DESCRICAO' in c.upper() and 'PONTO' in c.upper()][0]
    df[col_ete] = df[col_ete].astype(str).str.upper().str.strip().replace({'ETE 1':'ETE 01','ETE 2':'ETE 02'})
    col_data = 'DATA' if 'DATA' in df.columns else None

    conformidade, excedencias = [], []
    for p, par in params.items():
        if p not in df.columns:
            continue
        cat = par['categoria']
        obs = par.get('regra_alternativa') or par.get('condicao_adicional') or par.get('nota') or ''
        for ete in sorted(df[col_ete].dropna().unique()):
            sub = df[df[col_ete] == ete]
            serie = pd.to_numeric(sub[p], errors='coerce').dropna()
            n = len(serie)
            if n == 0:
                continue
            p95 = round(float(np.percentile(serie, 95)), 4)
            base = {'ETE':ete,'Parâmetro':par['rotulo'],'Categoria':CAT_LABEL[cat],'N':n,
                    'Limite':limite_str(par),'P95':p95,'Citação':par['citacao'],'Obs':obs}
            if cat == 'nao_exigivel':
                base.update({'Conformes':'-','Excedências':'-','% Conforme (amostral)':'-','Status':'Não exigível'})
                conformidade.append(base); continue
            ok = serie.apply(lambda v: conforme_amostra(v, par))
            n_ok = int(ok.sum()); n_exc = n - n_ok
            status = 'Conforme' if n_exc == 0 else f'NÃO CONFORME ({n_exc} excedência[s])'
            if cat == 'condicional_tabela1' and n_exc == 0:
                status = 'Conforme (condicional)'
            base.update({'Conformes':n_ok,'Excedências':n_exc,'% Conforme (amostral)':round(n_ok/n*100,1),'Status':status})
            conformidade.append(base)
            if n_exc > 0:
                for idx, val in serie[~ok].items():
                    d = sub.loc[idx, col_data] if col_data else None
                    excedencias.append({'ETE':ete,'Parâmetro':par['rotulo'],
                        'Data': pd.to_datetime(d).strftime('%m/%Y') if (col_data and pd.notna(d)) else '',
                        'Valor':round(float(val),4),'Limite':limite_str(par),'Categoria':CAT_LABEL[cat],'Citação':par['citacao']})

    df_conf = pd.DataFrame(conformidade)
    df_exc = pd.DataFrame(excedencias)
    df_rastr = pd.DataFrame(
        [{'Campo':k,'Valor':meta.get(k)} for k in ['id','titulo','publicacao','escopo_aplicado','fonte_arquivo','sha256_fonte','status','revisado_por','data_revisao']]
        + [{'Campo':'sha256_recalculado','Valor':sha_atual},{'Campo':'integridade','Valor':integridade}])

    caminho = DIR_PRODUTOS / "Relatorio_Conformidade_CONAMA_v4.xlsx"
    with pd.ExcelWriter(caminho, engine='openpyxl') as w:
        df_conf.to_excel(w, "Conformidade_Amostral", index=False)
        if not df_exc.empty:
            df_exc.to_excel(w, "Excedencias_Detalhe", index=False)
        df_rastr.to_excel(w, "Rastreabilidade_Norma", index=False)

    # --- Painel visual: conformidade amostral dos OBRIGATÓRIOS (Art. 21) ---
    obrig = [par['rotulo'] for p, par in params.items() if par['categoria'] == 'obrigatorio_art21' and p in df.columns]
    dob = df_conf[df_conf['Parâmetro'].isin(obrig) & (df_conf['% Conforme (amostral)'] != '-')].copy()
    if not dob.empty:
        dob['% Conforme (amostral)'] = pd.to_numeric(dob['% Conforme (amostral)'])
        piv = dob.pivot(index='Parâmetro', columns='ETE', values='% Conforme (amostral)')
        fig, ax = plt.subplots(figsize=(9, max(3, len(piv) * 0.75)))
        sns.heatmap(piv, annot=True, fmt='.0f', cmap='RdYlGn', vmin=0, vmax=100, linewidths=1.2, linecolor='white',
                    annot_kws={'fontweight':'bold'}, cbar_kws={'label':'% de amostras conformes','shrink':0.6}, ax=ax)
        ax.set_xlabel(''); ax.set_ylabel(''); ax.tick_params(length=0); plt.setp(ax.get_xticklabels(), fontweight='bold')
        ax.text(0,1.10,"Conformidade Amostral - CONAMA 430/11 (Art. 21, obrigatórios)",transform=ax.transAxes,fontsize=14,fontweight='bold',color='#2b2b2b',va='bottom',ha='left')
        ax.text(0,1.03,"% de amostras dentro do limite por ETE  |  fonte: PDF oficial (sha256 verificado)  |  revisão técnica pendente",transform=ax.transAxes,fontsize=9,color=NESA_COLORS['muted'],va='bottom',ha='left')
        fig.savefig(DIR_PRODUTOS / "Produto5_Conformidade_CONAMA.png"); plt.close(fig)

    # --- Resumo ---
    print(f"\n[+] Conformidade avaliada: {len(df_conf)} combinacoes ETE x parametro | excedencias: {len(df_exc)}")
    print(f"    Relatorio: {caminho.name}  |  Painel: Produto5_Conformidade_CONAMA.png")
    crit = df_conf[(df_conf['Categoria'] == CAT_LABEL['obrigatorio_art21']) &
                   (df_conf['Excedências'].apply(lambda x: isinstance(x, (int, np.integer)) and x > 0))]
    print("\n=== OBRIGATORIOS (Art. 21) COM EXCEDENCIA AMOSTRAL ===")
    print(crit[['ETE','Parâmetro','N','Excedências','% Conforme (amostral)','Limite']].to_string(index=False) if not crit.empty
          else "  Nenhuma excedencia nos parametros obrigatorios do Art. 21.")
    print("\n[!] Conformidade AMOSTRAL (Art. 24). Para DBO, o limite de 120 mg/L admite alternativa")
    print("    de >=60% de remocao (Art. 21, I, d) que NAO foi avaliada (requer DBO de entrada).")
except Exception as e:
    print(f"[!] Erro no motor de conformidade: {e}")


[*] Norma CONAMA-430-2011 | fonte: base_dados/CONAMA_n.430.2011.pdf | integridade: OK (hash confere)
[*] Status: extraido_do_pdf_oficial | revisado_por: None (revisao tecnica pendente)


C:\Users\thiago\AppData\Local\Temp\ipykernel_33844\3864881494.py:107: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df_conf.to_excel(w, "Conformidade_Amostral", index=False)
C:\Users\thiago\AppData\Local\Temp\ipykernel_33844\3864881494.py:109: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df_exc.to_excel(w, "Excedencias_Detalhe", index=False)
C:\Users\thiago\AppData\Local\Temp\ipykernel_33844\3864881494.py:110: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df_rastr.to_excel(w, "Rastreabilidade_Norma", index=False)



[+] Conformidade avaliada: 132 combinacoes ETE x parametro | excedencias: 30
    Relatorio: Relatorio_Conformidade_CONAMA_v4.xlsx  |  Painel: Produto5_Conformidade_CONAMA.png

=== OBRIGATORIOS (Art. 21) COM EXCEDENCIA AMOSTRAL ===
         ETE                      Parâmetro  N Excedências % Conforme (amostral) Limite
ETE COMPACTA                             pH 41           1                  97.6  5 a 9
ETE COMPACTA        Materiais Sedimentáveis  1           1                   0.0   <= 1
      ETE PM        Materiais Sedimentáveis  9           1                  88.9   <= 1
ETE COMPACTA Demanda Bioquímica de Oxigênio 41          10                  75.6 <= 120
      ETE 01                 Óleos e Graxas 38           1                  97.4 <= 100
      ETE PM                 Óleos e Graxas 44           1                  97.7 <= 100

[!] Conformidade AMOSTRAL (Art. 24). Para DBO, o limite de 120 mg/L admite alternativa
    de >=60% de remocao (Art. 21, I, d) que NAO foi avaliada (re